# Neuron Network - Lab

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, Exception):
    print("Local environment detected (KMUTT CPE342). Skipping Google Drive mount.")


Local environment detected (KMUTT CPE342). Skipping Google Drive mount.


### Part 1: Load  data

Import "bank-data.csv"

In [2]:
import pandas as pd
import os

data_path = 'bank-data.csv'
if not os.path.exists(data_path):
    data_path = '/content/drive/MyDrive/CPE_KMUTT/Year3/Semester_1-68/CPE342_MachineLearning/CPE342_Lab/CPE342_Lab_Assignment5_NeuralNetwork/bank-data.csv'
df = pd.read_csv(data_path, sep=';')
df


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no


### Part 2: Preprocess data

Preprocess the dataset as you have done before

#### 2.1 Binary encoding

Use LabelEncoder to encode the following columns:
- y
- default
- housing
- loan

In [3]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder instance
le = LabelEncoder()
df['y'] = le.fit_transform(df['y'])
df['default'] = le.fit_transform(df['default'])
df['housing'] = le.fit_transform(df['housing'])
df['loan'] = le.fit_transform(df['loan'])
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,oct,79,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,may,220,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,apr,185,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,jun,199,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,may,226,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,0,-333,1,0,cellular,30,jul,329,5,-1,0,unknown,0
4517,57,self-employed,married,tertiary,1,-3313,1,1,unknown,9,may,153,1,-1,0,unknown,0
4518,57,technician,married,secondary,0,295,0,0,cellular,19,aug,151,11,-1,0,unknown,0
4519,28,blue-collar,married,secondary,0,1137,0,0,cellular,6,feb,129,4,211,3,other,0


#### 2.2 Convert categorical variables into dummy columns

(1) Use pd.get_dummies to convert the following categorical variales into dummy columns
- job
- maritial
- education
- contact
- month
- poutcome

(2) Drop columns that have been converted

In [4]:
# Apply get_dummies to the following categorical column
df = pd.get_dummies(df, columns=['job', 'marital', 'education', 'contact', 'month', 'poutcome'])
df

,age,default,balance,housing,loan,day,duration,campaign,pdays,previous,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,30,0,1787,0,0,19,79,1,-1,0,...,False,False,False,False,True,False,False,False,False,True
1,33,0,4789,1,1,11,220,1,339,4,...,False,False,True,False,False,False,True,False,False,False
2,35,0,1350,1,0,16,185,1,330,1,...,False,False,False,False,False,False,True,False,False,False
3,30,0,1476,1,1,3,199,4,-1,0,...,True,False,False,False,False,False,False,False,False,True
4,59,0,0,1,0,5,226,1,-1,0,...,False,False,True,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,0,-333,1,0,30,329,5,-1,0,...,False,False,False,False,False,False,False,False,False,True
4517,57,1,-3313,1,1,9,153,1,-1,0,...,False,False,True,False,False,False,False,False,False,True
4518,57,0,295,0,0,19,151,11,-1,0,...,False,False,False,False,False,False,False,False,False,True
4519,28,0,1137,0,0,6,129,4,211,3,...,False,False,False,False,False,False,False,True,False,False


#### 2.3 Train/Test separation

Perform hold-out method
- 60% training set
- 40% testing set

##### X/y separation

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

#### 2.4 Feature Scaling

It is always a good practice to scale the features so that all of them can be uniformly evaluated

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Artificial Neural Network : sklearn

### Part 3: Train a model

In [7]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=1000)
mlp.fit(X_train, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(10, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


### Part 4: Model Evaluation

Evaluation metrics
- confusion metrix
- accuracy
- precision, recall, f1-score

In [8]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y_pred = mlp.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print('Accuracy: ', accuracy_score(y_test, y_pred))
print('Precision: ', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('F1-score: ', f1_score(y_test, y_pred))

[[1525   95]
 [ 115   74]]
Accuracy:  0.8839137645107794
Precision:  0.4378698224852071
Recall:  0.3915343915343915
F1-score:  0.4134078212290503


### Part 5: Model tuning

#### Note:

After building the classifier, try answering the following questions.

1. What is the Accuracy Score?
2. If you change your preprosessing method, can you improve the model?
3. If you change your parameters setting, can you improve the model?

In [9]:
# 1. What is the Accuracy Score?
print("=== Baseline MLPClassifier Evaluation ===")
print("Confusion Matrix:\n", cm)
print(f"Accuracy Score : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision Score: {precision_score(y_test, y_pred):.4f}")
print(f"Recall Score   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score       : {f1_score(y_test, y_pred):.4f}")


=== Baseline MLPClassifier Evaluation ===
Confusion Matrix:
 [[1525   95]
 [ 115   74]]
Accuracy Score : 0.8839
Precision Score: 0.4379
Recall Score   : 0.3915
F1-score       : 0.4134


#### 2. Preprocessing Method Comparison (StandardScaler vs. MinMaxScaler vs. RobustScaler)
Testing whether outlier handling or bounded feature scaling improves the decision boundary of MLPClassifier:

In [10]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

print("=== Preprocessing Comparison ===")
for name, sc in scalers.items():
    X_tr = sc.fit_transform(X_train)
    X_te = sc.transform(X_test)
    m = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    preds = m.predict(X_te)
    print(f"{name:15s} -> Accuracy: {accuracy_score(y_test, preds):.4f}, F1: {f1_score(y_test, preds):.4f}")


=== Preprocessing Comparison ===


StandardScaler  -> Accuracy: 0.8701, F1: 0.4169


MinMaxScaler    -> Accuracy: 0.8867, F1: 0.4730


RobustScaler    -> Accuracy: 0.8756, F1: 0.4094


#### 3. Parameter Tuning (Hidden Layers, Regularization, Early Stopping)
Experimenting with hidden layer topologies, Adam optimization, and L2 regularization:

In [11]:
param_configs = [
    {'name': 'Baseline (10, 10, 10)', 'hidden': (10, 10, 10), 'alpha': 0.0001, 'early_stopping': False},
    {'name': 'Deeper (64, 32)', 'hidden': (64, 32), 'alpha': 0.0001, 'early_stopping': False},
    {'name': 'Deeper (100, 50, 25)', 'hidden': (100, 50, 25), 'alpha': 0.001, 'early_stopping': False},
    {'name': 'Tuned (64, 32) + Reg + EarlyStop', 'hidden': (64, 32), 'alpha': 0.01, 'early_stopping': True},
]

print("=== Model Parameter Tuning ===")
for cfg in param_configs:
    clf = MLPClassifier(hidden_layer_sizes=cfg['hidden'], alpha=cfg['alpha'],
                        early_stopping=cfg['early_stopping'], max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    print(f"{cfg['name']:35s} -> Accuracy: {accuracy_score(y_test, preds):.4f}, F1: {f1_score(y_test, preds):.4f}")


=== Model Parameter Tuning ===


Baseline (10, 10, 10)               -> Accuracy: 0.8701, F1: 0.4169


Deeper (64, 32)                     -> Accuracy: 0.8795, F1: 0.4044


Deeper (100, 50, 25)                -> Accuracy: 0.8778, F1: 0.4260


Tuned (64, 32) + Reg + EarlyStop    -> Accuracy: 0.8972, F1: 0.4529


## Artificial Neural Network : keras
- See https://keras.io

Fitting a logistic regression model

### Part 3: Train a model

In [12]:
import keras; print(keras.__version__)

3.15.1


In [13]:
from keras import models
from keras import layers

In [14]:
X_train.shape

(2712, 48)

In [15]:
nn = models.Sequential()
nn.add(layers.Dense(48,activation = 'linear',input_shape=(None,48)))
nn.add(layers.Dense(1,activation = 'sigmoid'))

C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
nn.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

In [17]:
nn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, None, 48)       │         2,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 1)        │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,401 (9.38 KB)

 Trainable params: 2,401 (9.38 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
import numpy as np
X_train_add = np.expand_dims(X_train, axis=0)
y_train_add = np.expand_dims(y_train, axis=0)

In [19]:
history = nn.fit(X_train_add,y_train_add,epochs=100)

Epoch 1/100


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step - accuracy: 0.4395 - loss: 0.8402

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - accuracy: 0.4395 - loss: 0.8402


Epoch 2/100


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4440 - loss: 0.8323

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8595 - loss: 0.4743

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8595 - loss: 0.4743


Epoch 100/100


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8610 - loss: 0.4726

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8610 - loss: 0.4726


### Part 4: Model Evaluation

In [20]:
X_test_add = np.expand_dims(X_test, axis=0)
y_test_add = np.expand_dims(y_test, axis=0)

In [21]:
test_loss, test_acc = nn.evaluate(X_test_add, y_test_add)
print('Test Loss: %s\nTest Accuracy: %s' % (test_loss,test_acc))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.8662 - loss: 0.4591

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.8662 - loss: 0.4591


Test Loss: 0.45906102657318115
Test Accuracy: 0.8662244081497192


In [22]:
history.history

{'accuracy': [0.4395280182361603,
  0.4439527988433838,
  0.4528023600578308,
  0.45980826020240784,
  0.4609144628047943,
  0.46681416034698486,
  0.47418880462646484,
  0.4782448410987854,
  0.4845132827758789,
  0.48856931924819946,
  0.4933628439903259,
  0.4992625415325165,
  0.5055309534072876,
  0.5106931924819946,
  0.517699122428894,
  0.5232300758361816,
  0.5294985175132751,
  0.5357669591903687,
  0.5416666865348816,
  0.5516223907470703,
  0.5564159154891968,
  0.5623156428337097,
  0.5667403936386108,
  0.5722714066505432,
  0.5770648717880249,
  0.5873894095420837,
  0.5958701968193054,
  0.6043510437011719,
  0.6113569140434265,
  0.619837760925293,
  0.627212405204773,
  0.6345870494842529,
  0.6408554315567017,
  0.646386444568634,
  0.6519173979759216,
  0.6611356735229492,
  0.6699852347373962,
  0.6766223907470703,
  0.6836283206939697,
  0.6939527988433838,
  0.7002212405204773,
  0.7064896821975708,
  0.7112832069396973,
  0.7201327681541443,
  0.7252950072288513

### Part 5: Model tuning

#### Note:

After building the classifier, try answering the following questions.

1. What is the Accuracy Score?
2. If you change your preprosessing method, can you improve the model?
3. If you change your parameters setting, can you improve the model?

### Keras Model Tuning & Evaluation Answers

**1. What is the Accuracy Score?**
The baseline Keras logistic regression model achieves the accuracy printed below.

**2. Preprocessing & Batching Enhancements:**
Using 2D feature tensors directly with standard mini-batch training (`batch_size=64`) rather than expanding to 3D shape `(1, N, D)` improves vectorization and GPU/CPU cache efficiency.

**3. Hyperparameter & Architecture Tuning:**
Adding non-linear hidden layers (`Dense(64, relu)` + `Dense(32, relu)`), `Dropout(0.2)` for regularization, and the `Adam` optimizer provides substantial performance gains.

In [23]:
# Keras Model Tuning: Deep Neural Network Architecture
import keras
from keras import layers, models, optimizers

print(f"Baseline Keras Model Accuracy: {test_acc:.4f} (Loss: {test_loss:.4f})")

tuned_nn = models.Sequential([
    layers.Input(shape=(48,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

tuned_nn.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tuned_nn.summary()

tuned_history = tuned_nn.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

tuned_loss, tuned_acc = tuned_nn.evaluate(X_test, y_test, verbose=0)
print(f"\n=== Final Keras Comparison ===")
print(f"Baseline Model Accuracy: {test_acc:.4f}")
print(f"Tuned DNN Model Accuracy: {tuned_acc:.4f}")
print(f"Improvement: +{(tuned_acc - test_acc)*100:.2f}%")


Baseline Keras Model Accuracy: 0.8662 (Loss: 0.4591)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,249 (20.50 KB)

 Trainable params: 5,249 (20.50 KB)

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9433 - loss: 0.1420 - val_accuracy: 0.8913 - val_loss: 0.2883


Epoch 30/30


 1/34 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9375 - loss: 0.1260

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9424 - loss: 0.1371 - val_accuracy: 0.8913 - val_loss: 0.2860



=== Final Keras Comparison ===
Baseline Model Accuracy: 0.8662
Tuned DNN Model Accuracy: 0.9033
Improvement: +3.70%
